Production models are trained on the full RetailRocket dataset separately. This merge validates multi-source data handling.

## 1. Imports & Setup

In [1]:
# Core data processing
import pandas as pd
import numpy as np

# File handling
from pathlib import Path
import pyarrow.parquet as pq

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

pandas: 2.3.3
numpy: 2.3.5


---

## 2. Load Real Dataset (RetailRocket)

Load the cleaned RetailRocket dataset from notebook 01.

In [2]:
# Define path to RetailRocket data
RETAILROCKET_PATH = Path('artifacts/external/retailrocket/events.parquet')

if not RETAILROCKET_PATH.exists():
 raise FileNotFoundError(
 f"Error: RetailRocket dataset not found: {RETAILROCKET_PATH}\n"
 f"Please run notebook 01_retailrocket_eda.ipynb first to create this file."
 )

print(f" Loading real data from: {RETAILROCKET_PATH}")

# Load dataset
df_real = pd.read_parquet(RETAILROCKET_PATH)

# Add source column
df_real['source'] = 'retailrocket'

print(f"\n Loaded real dataset")
print(f" Shape: {df_real.shape}")
print(f" Columns: {list(df_real.columns)}")
print(f"\n Basic Stats:")
print(f" • Total events: {len(df_real):,}")
print(f" • Unique users: {df_real['user_id'].nunique():,}")
print(f" • Unique products: {df_real['product_id'].nunique():,}")
print(f" • Unique sessions: {df_real['session_id'].nunique():,}")
print(f"\n Event type distribution:")
print(df_real['event_type'].value_counts())

 Loading real data from: artifacts\external\retailrocket\events.parquet

 Loaded real dataset
 Shape: (2756101, 8)
 Columns: ['event_id', 'event_type', 'user_id', 'session_id', 'product_id', 'ts', 'properties', 'source']

 Basic Stats:
 • Total events: 2,756,101
 • Unique users: 1,407,580
 • Unique products: 235,061
 • Unique sessions: 1,666,974

 Event type distribution:
event_type
view           2664312
add_to_cart      69332
purchase         22457
Name: count, dtype: int64


---

## 3. Load Synthetic Dataset (P1 Simulator)

Load all synthetic event data generated by the P1 simulator.

In [7]:
# Define path to synthetic data
SYNTHETIC_BASE_PATH = Path('artifacts/events')

if not SYNTHETIC_BASE_PATH.exists():
 print(f"Warning: WARNING: Synthetic data directory not found: {SYNTHETIC_BASE_PATH}")
 print(f"Creating empty synthetic dataset for demonstration purposes.")
 print(f"To use real synthetic data, run the simulator first:\n")
 print(f" cd ../tools/simulator")
 print(f" python run_simulator.py --mode replay --events 10000 --output ../../notebooks/artifacts/events/sample.parquet\n")
 
 # Create a small synthetic dataset for testing
 df_synthetic = pd.DataFrame({
 'event_id': ['syn_' + str(i) for i in range(1000)],
 'event_type': np.random.choice(['view', 'click', 'add_to_cart', 'purchase'], 1000, p=[0.70, 0.15, 0.10, 0.05]),
 'user_id': ['user_syn_' + str(i) for i in np.random.randint(1, 100, 1000)],
 'session_id': ['session_syn_' + str(i) for i in np.random.randint(1, 200, 1000)],
 'product_id': ['prod_syn_' + str(i) for i in np.random.randint(1, 50, 1000)],
 'ts': pd.date_range('2024-01-01', periods=1000, freq='1min').strftime('%Y-%m-%dT%H:%M:%S+00:00'),
 'properties': ['{"source": "synthetic"}'] * 1000
 })
else:
 print(f" Scanning for synthetic data in: {SYNTHETIC_BASE_PATH}")
 
 # Find all parquet files recursively
 parquet_files = list(SYNTHETIC_BASE_PATH.rglob('*.parquet'))
 
 if len(parquet_files) == 0:
  print(f"Warning: WARNING: No parquet files found in {SYNTHETIC_BASE_PATH}")
  print(f"Creating empty synthetic dataset for demonstration.")
  df_synthetic = pd.DataFrame({
  'event_id': ['syn_' + str(i) for i in range(1000)],
  'event_type': np.random.choice(['view', 'click', 'add_to_cart', 'purchase'], 1000, p=[0.70, 0.15, 0.10, 0.05]),
  'user_id': ['user_syn_' + str(i) for i in np.random.randint(1, 100, 1000)],
  'session_id': ['session_syn_' + str(i) for i in np.random.randint(1, 200, 1000)],
  'product_id': ['prod_syn_' + str(i) for i in np.random.randint(1, 50, 1000)],
  'ts': pd.date_range('2024-01-01', periods=1000, freq='1min').strftime('%Y-%m-%dT%H:%M:%S+00:00'),
  'properties': ['{"source": "synthetic"}'] * 1000
  })
 else:
  print(f" Found {len(parquet_files)} parquet file(s)")
  print(f" Loading synthetic events...")
 
 # Load all files and concatenate
 dfs = []
 for pf in parquet_files:
  df_temp = pd.read_parquet(pf)
  dfs.append(df_temp)
  print(f" Loaded: {pf.name} ({len(df_temp):,} events)")
 
 df_synthetic = pd.concat(dfs, ignore_index=True)
 print(f"\n Concatenated {len(dfs)} file(s)")

# Add source column
df_synthetic['source'] = 'synthetic'

print(f"\n Loaded synthetic dataset")
print(f" Shape: {df_synthetic.shape}")
print(f" Columns: {list(df_synthetic.columns)}")
print(f"\n Basic Stats:")
print(f" • Total events: {len(df_synthetic):,}")
print(f" • Unique users: {df_synthetic['user_id'].nunique():,}")
print(f" • Unique products: {df_synthetic['product_id'].nunique():,}")
print(f" • Unique sessions: {df_synthetic['session_id'].nunique():,}")
print(f"\n Event type distribution:")
print(df_synthetic['event_type'].value_counts())

 Scanning for synthetic data in: artifacts\events
 Found 40 parquet file(s)
 Loading synthetic events...
 Loaded: 0146704c-9e33-482d-a958-05e86881c748.parquet (1 events)
 Loaded: 029df2e5-0fc9-4326-a2bd-9934c3861d1a.parquet (1 events)
 Loaded: 1de68abc-e9c6-47a3-8b80-bd4d2e39dd2f.parquet (1 events)
 Loaded: 2419f086-3a73-41dd-b3d4-f67b999dfb1b.parquet (1 events)
 Loaded: 438530ff-c656-4ae9-a114-f933f4d3ed71.parquet (1 events)
 Loaded: 4b2e8269-fa4f-493e-afdf-7f3ef1822949.parquet (1 events)
 Loaded: 4d51b0b6-6259-4428-9292-6ffad71994af.parquet (1 events)
 Loaded: 6ca9071f-fbe7-4230-a08b-7e77cfd44a7a.parquet (1 events)
 Loaded: 6d8061b8-a974-468e-abb3-fa647ab22387.parquet (1 events)
 Loaded: 721b2614-e8c1-4c3f-8dc0-6b2cd81032eb.parquet (1 events)
 Loaded: 750bad3a-6d6b-4148-b965-6678288831e5.parquet (1 events)
 Loaded: 8eb4c643-4d4d-45fc-ac41-be610a63cd26.parquet (1 events)
 Loaded: 92b916af-6d1b-4596-a08e-f91cfae6f1f1.parquet (1 events)
 Loaded: 9ad6eff0-8058-4618-8ec7-c6b21beeb9ea.parq

---

## 4. Schema Validation

**Critical Step**: Ensure both datasets have identical schemas before merging.

Required columns (internal event schema):
- `event_id`: Unique event identifier
- `event_type`: Action type (view, click, add_to_cart, purchase)
- `user_id`: User identifier
- `session_id`: Session identifier
- `product_id`: Product identifier
- `ts`: ISO 8601 timestamp
- `properties`: JSON metadata
- `source`: Data source (added in this notebook)

---

## 4. Schema Validation

Ensure both datasets have compatible schemas before merging.

In [8]:
print("Schema Validation\n")

# Check that both datasets have the same columns
real_cols = set(df_real.columns)
synthetic_cols = set(df_synthetic.columns)

# 1. Real dataset
print("1. REAL DATASET (RetailRocket):")
print(f"   Columns: {sorted(real_cols)}")

missing_in_real = synthetic_cols - real_cols
if missing_in_real:
    print(f"   Missing columns: {missing_in_real}")
    for col in missing_in_real:
        df_real[col] = None

extra_in_real = real_cols - synthetic_cols
if extra_in_real:
    print(f"   Extra columns (will be dropped): {extra_in_real}")
    df_real = df_real.drop(columns=list(extra_in_real))

# 2. Synthetic dataset
print("\n2. SYNTHETIC DATASET (P1 Simulator):")
print(f"   Columns: {sorted(synthetic_cols)}")

missing_in_synthetic = real_cols - synthetic_cols
if missing_in_synthetic:
    print(f"   Missing columns: {missing_in_synthetic}")
    for col in missing_in_synthetic:
        df_synthetic[col] = None

# 3. Data type validation
print("\n3. DATA TYPE VALIDATION:")
for col in sorted(set(df_real.columns) & set(df_synthetic.columns)):
    real_dtype = df_real[col].dtype
    synthetic_dtype = df_synthetic[col].dtype
    if real_dtype != synthetic_dtype:
        print(f"   {col}: real={real_dtype}, synthetic={synthetic_dtype}")
        if col in ['user_id', 'product_id', 'session_id']:
            df_real[col] = df_real[col].astype(str)
            df_synthetic[col] = df_synthetic[col].astype(str)

print("\nSchema validation complete")

Schema Validation

1. REAL DATASET (RetailRocket):
   Columns: ['event_id', 'event_type', 'product_id', 'properties', 'session_id', 'source', 'ts', 'user_id']

2. SYNTHETIC DATASET (P1 Simulator):
   Columns: ['event_id', 'event_type', 'product_id', 'properties', 'session_id', 'source', 'ts', 'user_id']

3. DATA TYPE VALIDATION:

Schema validation complete


In [9]:
# Define allowed event types
ALLOWED_EVENT_TYPES = {'view', 'click', 'add_to_cart', 'purchase'}

print("Event Type Normalization\n")

# Check real dataset
print("1. REAL DATASET:")
real_event_types = set(df_real['event_type'].unique())
print(f"   Event types: {real_event_types}")
unexpected_real = real_event_types - ALLOWED_EVENT_TYPES
if unexpected_real:
    print(f"   Unexpected event types: {unexpected_real}")
    print(f"   Dropping {len(df_real[df_real['event_type'].isin(unexpected_real)])} events...")
    df_real = df_real[df_real['event_type'].isin(ALLOWED_EVENT_TYPES)]
else:
    print(f"   All event types valid")

# Check synthetic dataset
print("\n2. SYNTHETIC DATASET:")
synthetic_event_types = set(df_synthetic['event_type'].unique())
print(f"   Event types: {synthetic_event_types}")
unexpected_synthetic = synthetic_event_types - ALLOWED_EVENT_TYPES
if unexpected_synthetic:
    print(f"   Unexpected event types: {unexpected_synthetic}")
    print(f"   Dropping {len(df_synthetic[df_synthetic['event_type'].isin(unexpected_synthetic)])} events...")
    df_synthetic = df_synthetic[df_synthetic['event_type'].isin(ALLOWED_EVENT_TYPES)]
else:
    print(f"   All event types valid")

print("\nEvent type normalization complete")

Event Type Normalization

1. REAL DATASET:
   Event types: {'purchase', 'add_to_cart', 'view'}
   All event types valid

2. SYNTHETIC DATASET:
   Event types: {'purchase', 'view', 'click', 'add_to_cart'}
   All event types valid

Event type normalization complete


---

## 6. Dataset Balancing

**Challenge**: Real dataset (RetailRocket) has 2.7M+ events, while synthetic may have fewer.

**Strategies**:
1. **No Balancing**: Keep all data (risk: model biased toward real data)
2. **Downsample Real**: Randomly sample real data to match synthetic size
3. **Upsample Synthetic**: Generate more synthetic data (requires re-running simulator)
4. **Weighted Sampling**: Keep all data, but weight by source during training

For this notebook, we'll compare sizes and apply **downsampling** if the imbalance is extreme (>10x).

In [10]:
print("Dataset Balancing Analysis\n")

real_count = len(df_real)
synthetic_count = len(df_synthetic)
ratio = real_count / synthetic_count if synthetic_count > 0 else float('inf')

print(f"Dataset Sizes:")
print(f"Real (RetailRocket): {real_count:,} events")
print(f"Synthetic (P1 Sim): {synthetic_count:,} events")
print(f"Ratio (Real:Synthetic): {ratio:.2f}:1\n")

# Decide on balancing strategy
if ratio > 10:
    print(f"IMBALANCE DETECTED: Real dataset is {ratio:.1f}x larger than synthetic")
    print(f"\nApplying downsampling to real dataset...\n")
    
    # Downsample to 2x synthetic size (preserve some real data advantage)
    target_real_size = min(real_count, synthetic_count * 2)
    
    if target_real_size < real_count:
        print(f"   Downsampling real dataset: {real_count:,} → {target_real_size:,}")
        df_real_balanced = df_real.sample(n=target_real_size, random_state=42)
        print(f"   Downsampled (random seed=42)")
    else:
        df_real_balanced = df_real
        print(f"   No downsampling needed (target size >= current size)")
    
    df_synthetic_balanced = df_synthetic
    
elif ratio < 0.1:
    print(f"IMBALANCE DETECTED: Synthetic dataset is {1/ratio:.1f}x larger than real")
    print(f"\nApplying downsampling to synthetic dataset...\n")
    
    # Downsample synthetic to 2x real size
    target_synthetic_size = min(synthetic_count, real_count * 2)
    
    if target_synthetic_size < synthetic_count:
        print(f"   Downsampling synthetic dataset: {synthetic_count:,} → {target_synthetic_size:,}")
        df_synthetic_balanced = df_synthetic.sample(n=target_synthetic_size, random_state=42)
        print(f"   Downsampled (random seed=42)")
    else:
        df_synthetic_balanced = df_synthetic
        print(f"   No downsampling needed")
    
    df_real_balanced = df_real
    
else:
    print(f"Dataset sizes are reasonably balanced (ratio < 10:1)")
    print(f"Keeping all data from both sources.\n")
    df_real_balanced = df_real
    df_synthetic_balanced = df_synthetic

print(f"\nFinal Balanced Sizes:")
print(f"Real: {len(df_real_balanced):,} events")
print(f"Synthetic: {len(df_synthetic_balanced):,} events")
print(f"Total: {len(df_real_balanced) + len(df_synthetic_balanced):,} events")

print("\nBalancing complete")

Dataset Balancing Analysis

Dataset Sizes:
Real (RetailRocket): 2,756,101 events
Synthetic (P1 Sim): 40 events
Ratio (Real:Synthetic): 68902.52:1

IMBALANCE DETECTED: Real dataset is 68902.5x larger than synthetic

Applying downsampling to real dataset...

   Downsampling real dataset: 2,756,101 → 80
   Downsampled (random seed=42)

Final Balanced Sizes:
Real: 80 events
Synthetic: 40 events
Total: 120 events

Balancing complete


---

## 7. Merge Datasets

Concatenate both datasets and sort by timestamp for temporal consistency.

In [11]:
print("Merging Datasets\n")

print(f"Pre-merge:")
print(f"Real: {len(df_real_balanced):,} events")
print(f"Synthetic: {len(df_synthetic_balanced):,} events\n")

# Concatenate datasets
df_combined = pd.concat([df_real_balanced, df_synthetic_balanced], ignore_index=True)

print(f"Concatenated: {len(df_combined):,} total events\n")

# Convert timestamp to datetime for sorting
# Use format='ISO8601' to handle both formats (with and without microseconds)
print(f"Converting timestamps to datetime...")
df_combined['ts_datetime'] = pd.to_datetime(df_combined['ts'], format='ISO8601')

# Sort by timestamp
print(f"Sorting by timestamp...")
df_combined = df_combined.sort_values('ts_datetime').reset_index(drop=True)

print(f"Sorted by timestamp")
print(f"Earliest event: {df_combined['ts_datetime'].min()}")
print(f"Latest event: {df_combined['ts_datetime'].max()}")
print(f"Time span: {(df_combined['ts_datetime'].max() - df_combined['ts_datetime'].min()).days} days")

# Drop temporary datetime column (keep original ts string)
df_combined = df_combined.drop(columns=['ts_datetime'])

print("\nDataset merge complete")

Merging Datasets

Pre-merge:
Real: 80 events
Synthetic: 40 events

Concatenated: 120 total events

Converting timestamps to datetime...
Sorting by timestamp...
Sorted by timestamp
Earliest event: 2015-05-04 05:40:49+00:00
Latest event: 2025-11-20 08:07:41.762415+00:00
Time span: 3853 days

Dataset merge complete


---

## 8. Final Validation

Verify the merged dataset integrity before saving.

In [12]:
print("Final Validation\n")

# 1. Total rows
print(f"1. DATASET SIZE:")
print(f"   Total events: {len(df_combined):,}")
print(f"   Expected: {len(df_real_balanced) + len(df_synthetic_balanced):,}")
assert len(df_combined) == len(df_real_balanced) + len(df_synthetic_balanced), "Row count mismatch!"
print(f"   Row count verified\n")

# 2. Unique counts
print(f"2. UNIQUE COUNTS:")
print(f"   Users: {df_combined['user_id'].nunique():,}")
print(f"   Products: {df_combined['product_id'].nunique():,}")
print(f"   Sessions: {df_combined['session_id'].nunique():,}")
print(f"   Event IDs: {df_combined['event_id'].nunique():,}")

# Check for duplicate event_ids
duplicate_event_ids = df_combined['event_id'].duplicated().sum()
if duplicate_event_ids > 0:
    print(f"   WARNING: {duplicate_event_ids} duplicate event_id values found!")
else:
    print(f"   All event_ids are unique\n")

# 3. Event type counts by source
print(f"3. EVENT TYPES BY SOURCE:")
event_type_by_source = pd.crosstab(df_combined['event_type'], df_combined['source'])
print(event_type_by_source)

# 4. Null value check
print(f"\n4. NULL VALUE CHECK:")
null_counts = df_combined.isnull().sum()
if null_counts.sum() == 0:
    print(f"   No null values")
else:
    print(f"   WARNING: Null values found:")
    print(null_counts[null_counts > 0])

# 5. Source distribution
print(f"\n5. SOURCE DISTRIBUTION:")
source_counts = df_combined['source'].value_counts()
print(source_counts)
print(f"\n   Real: {source_counts.get('retailrocket', 0):,} ({source_counts.get('retailrocket', 0)/len(df_combined)*100:.1f}%)")
print(f"   Synthetic: {source_counts.get('p1_simulator', 0):,} ({source_counts.get('p1_simulator', 0)/len(df_combined)*100:.1f}%)")

print("\nValidation complete")

Final Validation

1. DATASET SIZE:
   Total events: 120
   Expected: 120
   Row count verified

2. UNIQUE COUNTS:
   Users: 89
   Products: 85
   Sessions: 89
   Event IDs: 119
   All event_ids are unique

3. EVENT TYPES BY SOURCE:
source       retailrocket  synthetic
event_type                          
add_to_cart             4          4
click                   0          7
purchase                0          5
view                   76         24

4. NULL VALUE CHECK:
event_id    1
ts          1
dtype: int64

5. SOURCE DISTRIBUTION:
source
retailrocket    80
synthetic       40
Name: count, dtype: int64

   Real: 80 (66.7%)
   Synthetic: 0 (0.0%)

Validation complete


---

## 9. Save Combined Dataset

Save the merged dataset to a single Parquet file for downstream use.

In [13]:
import json

print("Converting properties column for Parquet compatibility...")

# Ensure all properties values are strings
df_combined['properties'] = df_combined['properties'].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else str(x) if x is not None else '{}'
)

print(f"Properties column converted")
print(f"All values are now strings")
print(f"Sample: {df_combined['properties'].iloc[0][:100]}")

Converting properties column for Parquet compatibility...
Properties column converted
All values are now strings
Sample: {"source": "retailrocket", "original_timestamp_ms": 1430718049441}


In [14]:
# Define output path
OUTPUT_DIR = Path('artifacts/combined')
OUTPUT_FILE = OUTPUT_DIR / 'events.parquet'

# Ensure directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving combined dataset...\n")
print(f"Output path: {OUTPUT_FILE}")
print(f"Rows: {len(df_combined):,}")
print(f"Columns: {list(df_combined.columns)}\n")

# Save as Parquet with compression
df_combined.to_parquet(OUTPUT_FILE, index=False, compression='snappy', engine='pyarrow')

print(f"Dataset saved successfully")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024**2:.2f} MB")

Saving combined dataset...

Output path: artifacts\combined\events.parquet
Rows: 120
Columns: ['event_id', 'event_type', 'user_id', 'session_id', 'product_id', 'ts', 'properties', 'source']

Dataset saved successfully
File size: 0.02 MB


In [15]:
# Verify saved file integrity
print("Verifying saved file...\n")

# Reload from disk
df_verify = pd.read_parquet(OUTPUT_FILE)

print(f"Successfully reloaded from disk")
print(f"Rows: {len(df_verify):,}")
print(f"Columns: {list(df_verify.columns)}")
print(f"\nMatch original: {len(df_verify) == len(df_combined)}")

if len(df_verify) == len(df_combined):
    print(f"\nFile integrity verified")
else:
    print(f"\nWARNING: Row count mismatch after reload")

Verifying saved file...

Successfully reloaded from disk
Rows: 120
Columns: ['event_id', 'event_type', 'user_id', 'session_id', 'product_id', 'ts', 'properties', 'source']

Match original: True

File integrity verified
